In [11]:
from sqlalchemy import text
import pandas as pd

def load_csv(table, path, engine):
    with engine.connect() as conn:
        conn.execute(text(f"TRUNCATE TABLE {table} RESTART IDENTITY CASCADE;"))
        conn.commit()

    df = pd.read_csv(path)
    df.to_sql(table, engine, if_exists='append', index=False)
    print(f"{table} loaded successfully")

In [12]:
load_csv("patients", "/app/data/raw/patients.csv",engine)

patients loaded successfully


In [13]:
load_csv("doctors", "/app/data/raw/doctors.csv",engine)

doctors loaded successfully


In [14]:
load_csv("appointments", "/app/data/raw/appointments.csv",engine)

appointments loaded successfully


In [16]:
load_csv("treatments", "/app/data/raw/treatments.csv",engine)

treatments loaded successfully


In [19]:
load_csv("billing", "/app/data/raw/billing.csv",engine)

billing loaded successfully


In [18]:
df = pd.read_csv("/app/data/raw/billing.csv")
df.to_sql("billing", engine, if_exists="replace", index=False)

200

In [21]:
import pandas as pd

patients = pd.read_csv("/app/data/raw/patients.csv")
doctors = pd.read_csv("/app/data/raw/doctors.csv")
appointments = pd.read_csv("/app/data/raw/appointments.csv")
treatments = pd.read_csv("/app/data/raw/treatments.csv")
billing = pd.read_csv("/app/data/raw/billing.csv")

In [22]:
for df, name in [(patients,"patients"),(doctors,"doctors"),
                 (appointments,"appointments"),
                 (treatments,"treatments"),
                 (billing,"billing")]:
    print(name)
    print(df.info())
    print(df.isnull().sum())
    print("Duplicates:", df.duplicated().sum())

patients
<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   patient_id          50 non-null     str  
 1   first_name          50 non-null     str  
 2   last_name           50 non-null     str  
 3   gender              50 non-null     str  
 4   date_of_birth       50 non-null     str  
 5   contact_number      50 non-null     int64
 6   address             50 non-null     str  
 7   registration_date   50 non-null     str  
 8   insurance_provider  50 non-null     str  
 9   insurance_number    50 non-null     str  
 10  email               50 non-null     str  
dtypes: int64(1), str(10)
memory usage: 4.4 KB
None
patient_id            0
first_name            0
last_name             0
gender                0
date_of_birth         0
contact_number        0
address               0
registration_date     0
insurance_provider    0
insurance_number    

In [23]:
patients.drop_duplicates(inplace=True)
doctors.drop_duplicates(inplace=True)
appointments.drop_duplicates(inplace=True)
treatments.drop_duplicates(inplace=True)
billing.drop_duplicates(inplace=True)

In [24]:
patients['date_of_birth'] = pd.to_datetime(patients['date_of_birth'])
patients['registration_date'] = pd.to_datetime(patients['registration_date'])

appointments['appointment_date'] = pd.to_datetime(appointments['appointment_date'])
appointments['appointment_time'] = pd.to_datetime(appointments['appointment_time'], format='%H:%M:%S').dt.time

treatments['treatment_date'] = pd.to_datetime(treatments['treatment_date'])
billing['bill_date'] = pd.to_datetime(billing['bill_date'])

In [25]:
appointments['status'] = appointments['status'].str.strip().str.title()
patients['gender'] = patients['gender'].str.strip().str.title()
billing['payment_status'] = billing['payment_status'].str.strip().str.title()

In [27]:
# Patients
patients = patients.fillna({
    'insurance_provider': 'Unknown',
    'insurance_number': 'NA'
})

# Doctors
doctors = doctors.fillna({
    'email': 'Not Provided'
})

# Billing
billing['payment_method'] = billing['payment_method'].fillna('Unknown')
billing['payment_status'] = billing['payment_status'].fillna('Unknown')

In [28]:
today = pd.Timestamp.today()
patients['age'] = (today - patients['date_of_birth']).dt.days // 365

In [29]:
appointments['appointment_month'] = appointments['appointment_date'].dt.to_period('M')

In [30]:
treatments['is_expensive'] = treatments['cost'] > treatments['cost'].mean()

In [31]:
master = appointments.merge(patients, on='patient_id') \
                     .merge(doctors, on='doctor_id') \
                     .merge(treatments, on='appointment_id') \
                     .merge(billing, on=['patient_id','treatment_id'])

In [ ]:
patients.to_csv("/appdata/cleaned/patients_cleaned.csv", index=False)
doctors.to_csv("/appdata/cleaned/doctors_cleaned.csv", index=False)
appointments.to_csv("/appdata/cleaned/appointments_cleaned.csv", index=False)
treatments.to_csv("data/cleaned/treatments_cleaned.csv", index=False)
billing.to_csv("data/cleaned/billing_cleaned.csv", index=False)
master.to_csv("data/cleaned/master_table.csv", index=False)